In [3]:
import pandas as pd

Perth_Cycling_df = pd.read_csv('Perth_Cycling_df.csv')
df = Perth_Cycling_df.copy()  # Create a copy of the cycling-station subsample for further analysis
print(f"Loaded Perth_Cycling_df: {len(df)} rows")


Loaded Perth_Cycling_df: 32703 rows


In [4]:
import math


def coordinate_range(longtitude, latitude, radius_km):
    """Return the latitude/longitude bounds within a radius in kilometres."""
    earth_radius_km = 6371.0
    lat_delta = math.degrees(radius_km / earth_radius_km)
    cos_lat = math.cos(math.radians(latitude))
    lon_delta = math.degrees(radius_km / (earth_radius_km * cos_lat)) if cos_lat != 0 else float('inf')

    return {
        'min_longitude': longtitude - lon_delta,
        'max_longitude': longtitude + lon_delta,
        'min_latitude': latitude - lat_delta,
        'max_latitude': latitude + lat_delta,
    }

# Reapply neighbour counts using the coordinates already in df
station_lookup = df[['TRADING_NAME', 'Latitude', 'Longitude']].drop_duplicates().dropna(subset=['Latitude', 'Longitude']).reset_index(drop=True)
neighbor_counts = []
for _, station_row in station_lookup.iterrows():
    bounds = coordinate_range(station_row['Longitude'], station_row['Latitude'], 5)
    neighbor_counts.append({
        'TRADING_NAME': station_row['TRADING_NAME'],
        'neighbour': station_lookup[
            (station_lookup['TRADING_NAME'] != station_row['TRADING_NAME']) &
            station_lookup['Latitude'].between(bounds['min_latitude'], bounds['max_latitude']) &
            station_lookup['Longitude'].between(bounds['min_longitude'], bounds['max_longitude'])
        ].shape[0]
    })

neighbor_counts_df = pd.DataFrame(neighbor_counts)
df = df.merge(neighbor_counts_df, on='TRADING_NAME', how='left')

# Create a follow dummy: 1 if any nearby station within 5 km has relented in the current cycle, else 0
df['PUBLISH_DATE'] = pd.to_datetime(df['PUBLISH_DATE'])
df = df.sort_values(['TRADING_NAME', 'PUBLISH_DATE']).reset_index(drop=True)
df['station_price_change'] = df.groupby('TRADING_NAME')['PRODUCT_PRICE'].diff()
df['relented'] = (df['station_price_change'] > 10).astype(int)
df['cycle_id'] = df.groupby('TRADING_NAME')['relented'].cumsum()

neighbor_map = {}
for _, station_row in station_lookup.iterrows():
    bounds = coordinate_range(station_row['Longitude'], station_row['Latitude'], 5)
    neighbor_map[station_row['TRADING_NAME']] = station_lookup[
        (station_lookup['TRADING_NAME'] != station_row['TRADING_NAME']) &
        station_lookup['Latitude'].between(bounds['min_latitude'], bounds['max_latitude']) &
        station_lookup['Longitude'].between(bounds['min_longitude'], bounds['max_longitude'])
    ]['TRADING_NAME'].tolist()

relented_events = df.loc[df['relented'] == 1, ['TRADING_NAME', 'PUBLISH_DATE', 'cycle_id']].copy()
relented_events = relented_events.sort_values(['TRADING_NAME', 'cycle_id', 'PUBLISH_DATE'])
relented_by_station_cycle = {
    station: {
        cycle_id: group['PUBLISH_DATE'].tolist()
        for cycle_id, group in station_group.groupby('cycle_id')
    }
    for station, station_group in relented_events.groupby('TRADING_NAME')
}

def has_neighbor_relented(row):
    current_cycle = row['cycle_id']
    current_date = row['PUBLISH_DATE']
    for neighbor in neighbor_map.get(row['TRADING_NAME'], []):
        cycle_dates = relented_by_station_cycle.get(neighbor, {}).get(current_cycle, [])
        if any(date <= current_date for date in cycle_dates):
            return 1
    return 0

df['follow'] = df.apply(has_neighbor_relented, axis=1)

neighbour = int(df['follow'].sum())
print(f"Rows with follow=1: {neighbour}")
print(f"Share follow=1: {df['follow'].mean():.4f}")
print(df[['TRADING_NAME', 'Latitude', 'Longitude', 'neighbour']].drop_duplicates().head(10))

Rows with follow=1: 29873
Share follow=1: 0.9135
                  TRADING_NAME  Latitude  Longitude  neighbour
0            7-Eleven Balcatta  -31.8845   115.7985         27
365     7-Eleven Banksia Grove  -31.7050   115.7940         15
730            7-Eleven Butler  -31.6485   115.7195         12
1095        7-Eleven Joondalup  -31.7440   115.7615         19
1460         7-Eleven Stirling  -31.8665   115.8210         18
1825      Ampol Foodary Butler  -31.6350   115.6970          8
2190      Ampol Foodary Carine  -31.8355   115.7705         14
2555    Ampol Foodary Clarkson  -31.7015   115.7135         14
2920  Ampol Foodary Doubleview  -31.8930   115.7810         25
3285  Ampol Foodary East Perth  -31.9550   115.8740         12


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# 1. Parse date column and sort for station-level time ordering
df['PUBLISH_DATE'] = pd.to_datetime(df['PUBLISH_DATE'])
df = df.sort_values(['TRADING_NAME', 'PUBLISH_DATE']).reset_index(drop=True)

# 2. Create event indicator: E_t = 1 if date is June 1st or later, 0 otherwise
event_date = pd.Timestamp('2022-06-01')
df['event_indicator'] = (df['PUBLISH_DATE'] >= event_date).astype(int)

print(f"Event date: {event_date}")
print(f"Number of records with event_indicator=1: {(df['event_indicator'] == 1).sum()}")
print(f"Number of records with event_indicator=0: {(df['event_indicator'] == 0).sum()}")

# 3. Identify major brands (top 5 most frequent)
brand_counts = df['BRAND_DESCRIPTION'].value_counts()
major_brands = brand_counts.head(5).index.tolist()
print(f"\nMajor brands (top 5): {major_brands}")

# 4. Create major brand indicator: 1 if brand is in top 5, 0 otherwise
df['major_brand_indicator'] = df['BRAND_DESCRIPTION'].isin(major_brands).astype(int)

print(f"\nMajor brand indicator created")
print(f"  Records with major brand: {(df['major_brand_indicator'] == 1).sum()}")
print(f"  Records with other brands: {(df['major_brand_indicator'] == 0).sum()}")

# 5. Create pooled station-level delta retail price and position variables
# delta_retail_price = today's retail price minus yesterday's retail price for the same station
# position = yesterday's retail price minus today's TGP

df['yesterday_price'] = df.groupby('TRADING_NAME')['PRODUCT_PRICE'].shift(1)
df['delta_retail_price'] = df.groupby('TRADING_NAME')['PRODUCT_PRICE'].diff()
df['position'] = df['yesterday_price'] - df['TGP_Perth']

print(f"\nDelta retail price and position created")
print(f"  Non-missing delta_retail_price: {df['delta_retail_price'].notna().sum()}")
print(f"  Non-missing position: {df['position'].notna().sum()}")

# 6. Display summary statistics
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns related to TVTP specification:")
print(f"  - event_indicator: {df['event_indicator'].unique()}")
print(f"  - major_brand_indicator: {df['major_brand_indicator'].unique()}")

# 7. Preview the data with new variables
print(f"\nSample of pooled station-level variables:")
print(df[['TRADING_NAME', 'PUBLISH_DATE', 'PRODUCT_PRICE', 'yesterday_price', 'delta_retail_price', 'position', 'TGP_Perth', 'major_brand_indicator']].head(10))

Event date: 2022-06-01 00:00:00
Number of records with event_indicator=1: 19113
Number of records with event_indicator=0: 13590

Major brands (top 5): ['BP', 'Ampol', 'Coles Express', 'Puma', 'United']

Major brand indicator created
  Records with major brand: 26311
  Records with other brands: 6392

Delta retail price and position created
  Non-missing delta_retail_price: 32609
  Non-missing position: 32609

DataFrame shape: (32703, 25)

Columns related to TVTP specification:
  - event_indicator: [0 1]
  - major_brand_indicator: [0 1]

Sample of pooled station-level variables:
        TRADING_NAME PUBLISH_DATE  PRODUCT_PRICE  yesterday_price  \
0  7-Eleven Balcatta   2022-01-01          162.5              NaN   
1  7-Eleven Balcatta   2022-01-02          169.9            162.5   
2  7-Eleven Balcatta   2022-01-03          159.7            169.9   
3  7-Eleven Balcatta   2022-01-04          156.7            159.7   
4  7-Eleven Balcatta   2022-01-05          151.9            156.7   
5

In [6]:
# ============================================================================
# Step 3: Prepare exog_tvtp matrix with Position, Ramp, Major, Follow, Neighbour, and Interactions
# ============================================================================
# 
# Specification (Multinomial Logit form):
#   P(S_t = j | S_{t-1} = i, Z_t) = exp(Z_t' γ_ij) / Σ_k exp(Z_t' γ_ik)
#
# where Z_t = [1, position_t, ramp_t, major_t, follow_t, neighbour_t, ramp_t × major_t, ramp_t × neighbour_t]
#   - position_t = yesterday price - today's TGP
#   - ramp_t = 0 if t < June 1st; = (t - June 1st).days if t >= June 1st
#   - major_t = 1 if major brand, 0 otherwise
#   - follow_t = 1 if any nearby station within 5 km has relented in the current cycle, 0 otherwise
#   - neighbour_t = number of nearby stations within 5 km
#   - ramp_t × major_t captures how the effect of ramp differs for major brands
#   - ramp_t × neighbour_t captures how the effect of ramp differs with local station density

from statsmodels.tools.tools import add_constant

# Keep pooled station-level observations that define the endogenous variable
pooled_df = df.dropna(subset=['delta_retail_price', 'position', 'major_brand_indicator', 'follow', 'neighbour']).copy()
pooled_df = pooled_df.sort_values(['PUBLISH_DATE', 'TRADING_NAME']).reset_index(drop=True)

# Create ramp variable: days since June 1st (0 before June 1st)
base_date = pd.Timestamp('2022-06-01')
pooled_df['ramp_variable'] = ((pooled_df['PUBLISH_DATE'] - base_date).dt.days).clip(lower=0)

# Create interaction terms
pooled_df['ramp_x_major'] = pooled_df['ramp_variable'] * pooled_df['major_brand_indicator']
pooled_df['ramp_x_neighbour'] = pooled_df['ramp_variable'] * pooled_df['neighbour']

# Build exog_tvtp matrix: constant + position + ramp + major + follow + neighbour + interactions
exog_tvtp = add_constant(
    pooled_df[['position', 'ramp_variable', 'major_brand_indicator', 'follow', 'neighbour', 'ramp_x_major', 'ramp_x_neighbour']], 
    prepend=True
)
exog_tvtp.columns = ['const', 'position_t', 'ramp_t', 'major_t', 'follow_t', 'neighbour_t', 'ramp_t_x_major', 'ramp_t_x_neighbour']

print("=" * 70)
print("EXOGENOUS VARIABLES FOR TIME-VARYING TRANSITION PROBABILITIES")
print("=" * 70)
print(f"\nPooled observation count: {len(pooled_df)}")
print(f"Exog_TVTP specification (Multinomial Logit form):")
print(f"  Z_t = [1 (constant), position_t, ramp_t, major_t, follow_t, neighbour_t, ramp_t × major_t, ramp_t × neighbour_t]")
print(f"\nWhere:")
print(f"  position_t = yesterday price - today's TGP")
print(f"  ramp_t = 0 if PUBLISH_DATE < 2022-06-01")
print(f"  ramp_t = (PUBLISH_DATE - 2022-06-01).days if PUBLISH_DATE >= 2022-06-01")
print(f"  major_t = 1 if major brand, 0 otherwise")
print(f"  follow_t = 1 if any nearby station within 5 km has relented in the current cycle")
print(f"  neighbour_t = number of nearby stations within 5 km")
print(f"  ramp_t × major_t = ramp effect modulation by brand type")
print(f"  ramp_t × neighbour_t = ramp effect modulation by local station density")
print(f"\nExog_TVTP shape: {exog_tvtp.shape}")
print(f"Exog_TVTP columns: {list(exog_tvtp.columns)}")
print(f"\nDate range: {pooled_df['PUBLISH_DATE'].min()} to {pooled_df['PUBLISH_DATE'].max()}")

# Display sample of exog_tvtp
print(f"\nFirst 10 rows of exog_tvtp:")
print(exog_tvtp.head(10))

print(f"\nRows around event date (2022-06-01):")
event_idx = pooled_df[pooled_df['PUBLISH_DATE'] == pd.Timestamp('2022-06-01')].index[0]
print(exog_tvtp.iloc[max(0, event_idx-3):min(len(exog_tvtp), event_idx+5)])

print(f"\nRamp variable statistics:")
print(f"  Min: {pooled_df['ramp_variable'].min()}")
print(f"  Max: {pooled_df['ramp_variable'].max()}")
print(f"  Mean (post-event): {pooled_df[pooled_df['ramp_variable'] > 0]['ramp_variable'].mean():.1f}")

print(f"\nMajor brand distribution in exog_tvtp:")
print(f"  Share with major_t=1: {pooled_df['major_brand_indicator'].mean():.4f}")
print(f"  Share with major_t=0: {(1 - pooled_df['major_brand_indicator'].mean()):.4f}")
print(f"  Share with follow_t=1: {pooled_df['follow'].mean():.4f}")
print(f"  Mean neighbour count: {pooled_df['neighbour'].mean():.2f}")

EXOGENOUS VARIABLES FOR TIME-VARYING TRANSITION PROBABILITIES

Pooled observation count: 32609
Exog_TVTP specification (Multinomial Logit form):
  Z_t = [1 (constant), position_t, ramp_t, major_t, follow_t, neighbour_t, ramp_t × major_t, ramp_t × neighbour_t]

Where:
  position_t = yesterday price - today's TGP
  ramp_t = 0 if PUBLISH_DATE < 2022-06-01
  ramp_t = (PUBLISH_DATE - 2022-06-01).days if PUBLISH_DATE >= 2022-06-01
  major_t = 1 if major brand, 0 otherwise
  follow_t = 1 if any nearby station within 5 km has relented in the current cycle
  neighbour_t = number of nearby stations within 5 km
  ramp_t × major_t = ramp effect modulation by brand type
  ramp_t × neighbour_t = ramp effect modulation by local station density

Exog_TVTP shape: (32609, 8)
Exog_TVTP columns: ['const', 'position_t', 'ramp_t', 'major_t', 'follow_t', 'neighbour_t', 'ramp_t_x_major', 'ramp_t_x_neighbour']

Date range: 2022-01-02 00:00:00 to 2022-12-31 00:00:00

First 10 rows of exog_tvtp:
   const  positi

In [7]:
# ============================================================================
# Step 4: Prepare endogenous variable and fit pooled TVTP Markov Switching Model
# ============================================================================

# Pooled station-level endogenous variable: delta retail price
endog = pooled_df['delta_retail_price'].values

# Mean-equation exogenous regressors: position, major brand indicator, and interaction
pooled_df['position_x_major'] = pooled_df['position'] * pooled_df['major_brand_indicator']
exog_mean = pooled_df[['position', 'follow']]

print("=" * 70)
print("ENDOGENOUS VARIABLE PREPARATION")
print("=" * 70)
print(f"\nPooled station-level data shape: {pooled_df.shape}")
print(f"Endogenous variable (delta_retail_price) shape: {endog.shape}")
print(f"Exogenous mean-equation variables shape: {exog_mean.shape}")
print(f"Date range matches exog_tvtp: {len(endog) == len(exog_tvtp)}")

# Display summary
print(f"\nDelta retail price statistics:")
print(f"  Mean: {pooled_df['delta_retail_price'].mean():.4f}")
print(f"  Median: {pooled_df['delta_retail_price'].median():.4f}")
print(f"  Std: {pooled_df['delta_retail_price'].std():.4f}")
print(f"  Min: {pooled_df['delta_retail_price'].min():.4f}")
print(f"  Max: {pooled_df['delta_retail_price'].max():.4f}")

print(f"\nMean-equation covariate statistics:")
print(f"  position mean: {pooled_df['position'].mean():.4f}")
print(f"  major brand share: {pooled_df['major_brand_indicator'].mean():.4f}")
print(f"  position x major mean: {pooled_df['position_x_major'].mean():.4f}")

print(f"\nSample of pooled station-level data:")
print(pooled_df[['TRADING_NAME', 'PUBLISH_DATE', 'PRODUCT_PRICE', 'yesterday_price', 'delta_retail_price', 'position', 'major_brand_indicator', 'position_x_major', 'ramp_variable']].head(10))

# ============================================================================
# Step 5: Fit TVTP Markov Regime-Switching Model
# ============================================================================

from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

print("\n" + "=" * 70)
print("FITTING POOLED TIME-VARYING TRANSITION PROBABILITY MARKOV MODEL")
print("=" * 70)

# Model specification:
# - 2 regimes
# - Endogenous variable: delta retail price
# - Mean-equation exogenous regressors: position + follow
# - Exogenous TVTP: constant + position + ramp + ramp-position interaction

try:
    model_tvtp = MarkovRegression(
        endog=endog,
        k_regimes=2,
        trend='c',
        exog=exog_mean,
        exog_tvtp=exog_tvtp,
        switching_trend=True,
        switching_exog=True,
        switching_variance=True
    )
    
    print("\nModel initialized successfully!")
    print(f"  Number of regimes: {model_tvtp.k_regimes}")
    print(f"  Pooled station-level observations: {len(endog)}")
    print(f"  Mean-equation exogenous variables: {exog_mean.shape[1]}")
    print(f"  TVTP exogenous variables: {exog_tvtp.shape[1]}")
    print(f"  Mean equation: c + position + follow")
    print(f"  TVTP specification: constant + position + ramp + ramp-position interaction")
    
    print("\nFitting model via Maximum Likelihood Estimation...")
    results_tvtp = model_tvtp.fit(disp=False, method='powell', maxiter=20, em_iter=5, search_reps=0)
    
    print("\n✓ Model fitted successfully!")
    print(f"\nFitted model summary:")
    print(f"  Log-likelihood: {results_tvtp.llf:.2f}")
    print(f"  AIC: {results_tvtp.aic:.2f}")
    print(f"  BIC: {results_tvtp.bic:.2f}")
    
except Exception as e:
    print(f"\n✗ Error fitting model: {str(e)}")
    import traceback
    traceback.print_exc()

ENDOGENOUS VARIABLE PREPARATION

Pooled station-level data shape: (32609, 29)
Endogenous variable (delta_retail_price) shape: (32609,)
Exogenous mean-equation variables shape: (32609, 2)
Date range matches exog_tvtp: True

Delta retail price statistics:
  Mean: -0.0058
  Median: -2.0000
  Std: 10.8420
  Min: -40.0000
  Max: 50.0000

Mean-equation covariate statistics:
  position mean: 12.8339
  major brand share: 0.8046
  position x major mean: 10.9510

Sample of pooled station-level data:
               TRADING_NAME PUBLISH_DATE  PRODUCT_PRICE  yesterday_price  \
0         7-Eleven Balcatta   2022-01-02          169.9            162.5   
1    7-Eleven Banksia Grove   2022-01-02          169.5            174.9   
2           7-Eleven Butler   2022-01-02          171.9            174.9   
3        7-Eleven Joondalup   2022-01-02          171.9            159.9   
4         7-Eleven Stirling   2022-01-02          159.9            168.9   
5      Ampol Foodary Butler   2022-01-02         

In [9]:
# ============================================================================
# Step 6: Extract and Analyze Regime-Switching Results
# ============================================================================

from stargazer.stargazer import Stargazer

print("=" * 70)
print("REGIME-SPECIFIC PARAMETERS")
print("=" * 70)

params = results_tvtp.params
print(f"\nModel parameters ({len(params)} total):")
print(results_tvtp.summary())


stargazer_table = Stargazer([results_tvtp])
stargazer_table.title('TVTP Markov Switching Model Results')
stargazer_table.show_model_numbers(False)
stargazer_table.significance_levels([0.1, 0.05, 0.01])
latex_output = stargazer_table.render_latex()
with open('results_tvtp_summary_table.tex', 'w', encoding='utf-8') as latex_file:
    latex_file.write(latex_output)
print('Saved LaTeX table to results_tvtp_summary_table.tex using Stargazer')

REGIME-SPECIFIC PARAMETERS

Model parameters (24 total):
                        Markov Switching Model Results                        
Dep. Variable:                      y   No. Observations:                32609
Model:               MarkovRegression   Log Likelihood              -89617.998
Date:                Tue, 23 Jun 2026   AIC                         179283.996
Time:                        01:13:13   BIC                         179485.413
Sample:                             0   HQIC                        179348.368
                              - 32609                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.3591      0.051    -26.457      0.000      -1.460      -1

AttributeError: 'numpy.ndarray' object has no attribute 'index'